In [0]:
from pyspark.sql import functions as F

In [0]:
storage_account = "dlsua5816bd"
login = "mariiasirska" 
container = login

catalog = "dbr_dev_ua5816bd"
schema = f"{login}_bronze"
 
base = f"abfss://{container}@{storage_account}.dfs.core.windows.net/lab3"
landing = f"{base}/landing/orders"
schema_loc = f"{base}/_schemas/orders"
checkpoint = f"{base}/_checkpoints/orders"
table = f"`{catalog}`.`{schema}`.lab3_orders_bronze"

Generating files:

In [0]:
def gen_orders(start, n, files, rename_customer=False, extra_col=False):
    df = (
        spark.range(start, start + n).withColumnRenamed("id", "order_id")
        .withColumn("customer_id", (F.rand() * 1000).cast("int"))
        .withColumn("amount", F.round(F.rand() * 500, 2))
        .withColumn(
            "status",
            F.element_at(F.array(F.lit("new"), F.lit("paid"), F.lit("shipped")),
                         (F.floor(F.rand() * 3) + 1).cast("int")),
        )
        .withColumn("created_at", F.date_format(F.current_timestamp(), "yyyy-MM-dd'T'HH:mm:ss"))
    )
    if rename_customer:
        df = df.withColumnRenamed("customer_id", "client_id")
    if extra_col:
        df = df.withColumn("channel", F.lit("web"))
    df.repartition(files).write.mode("append").json(landing)
 
gen_orders(start=0, n=100_000, files=1000)
print("files in landing:", len([f for f in dbutils.fs.ls(landing) if f.name.endswith(".json")]))

Auto Loader stream into bronze:

In [0]:
def start_stream(trigger=None, max_files=None, evolution="addNewColumns"):
    reader = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_loc)
        .option("cloudFiles.schemaEvolutionMode", evolution)
        .option("cloudFiles.inferColumnTypes", "true")
    )
    if max_files:
        reader = reader.option("cloudFiles.maxFilesPerTrigger", max_files)
 
    df = (
        reader.load(landing)
        .withColumn("source_file", F.col("_metadata.file_path"))
        .withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("load_date", F.current_date())
    )
    return (
        df.writeStream
        .option("checkpointLocation", checkpoint)
        .option("mergeSchema", "true")
        .trigger(**(trigger or {"availableNow": True}))
        .toTable(table)
    )

In [0]:
q = start_stream(max_files=100)
q.awaitTermination()

In [0]:
spark.table(table).printSchema()
display(spark.table(table).limit(10))

Streaming statistics:

In [0]:

for p in q.recentProgress:
    m = p["sources"][0]["metrics"]
    print(p["batchId"], "rows:", p["numInputRows"],
          "| outstanding files:", m.get("numFilesOutstanding"),
          "| outstanding bytes:", m.get("numBytesOutstanding"))
 

In [0]:
print("distinct source files:", spark.table(table).select("source_file").distinct().count())
print("rows:", spark.table(table).count())

Rescued data column:

In [0]:
(spark.range(10)
 .selectExpr("id + 900000 as order_id", "1 as customer_id", "'N/A' as amount",
             "'new' as status", "date_format(current_timestamp(), \"yyyy-MM-dd'T'HH:mm:ss\") as created_at")
 .coalesce(1).write.mode("append").json(landing))
 
q = start_stream(); q.awaitTermination()
 
display(spark.table(table).filter("_rescued_data IS NOT NULL"))
 

Schema evolution. New column "channel":

In [0]:
gen_orders(start=1_000_000, n=1_000, files=10, extra_col=True)

try:
    q = start_stream(); q.awaitTermination()
except Exception as e:
    print("Stream failed as expected:", type(e).__name__, str(e)[:300])

In [0]:
q = start_stream(); q.awaitTermination()
spark.table(table).printSchema()
display(spark.table(table).filter("channel IS NOT NULL").limit(10))
 

Schema evolution: renamed column

In [0]:
gen_orders(start=2_000_000, n=1_000, files=10, rename_customer=True)
 
try:
    q = start_stream(); q.awaitTermination()
except Exception as e:
    print("Stream failed as expected:", type(e).__name__, str(e)[:300])

In [0]:
q = start_stream(); q.awaitTermination()

display(spark.table(table).select("order_id", "customer_id", "client_id").orderBy(F.desc("order_id")).limit(10))
 

In [0]:
display(dbutils.fs.ls(f"{schema_loc}/_schemas"))

Trigger types:

In [0]:
gen_orders(start=3_000_000, n=5_000, files=50)
q = start_stream(trigger={"availableNow": True}); q.awaitTermination()

In [0]:
import time
gen_orders(start=5_000_000, n=5_000, files=50)
q = start_stream(trigger={"processingTime": "10 seconds"})
time.sleep(60)
q.stop()

Safe reload:

In [0]:
before = spark.table(table).count()
q = start_stream(); q.awaitTermination()
print("no duplicates on re-run:", before == spark.table(table).count())

In [0]:
for s in spark.streams.active:
    s.stop()
 
dbutils.fs.rm(checkpoint, True)
spark.sql(f"TRUNCATE TABLE {table}")
 
q = start_stream(max_files=200); q.awaitTermination()
 
print("rows:", spark.table(table).count())
print("distinct source files:", spark.table(table).select("source_file").distinct().count())